# Solving Systems of Equations with More Variables

*Course notes for **Math for Machine Learning**, C1 · W2 · L1 — "Solving Systems of Equations with More Variables" (DeepLearning.AI).*

The elimination method scales up. With three unknowns the idea is the same, applied recursively:

1. Use the first equation to **remove the first variable** from all the others.
2. What remains is a smaller system (here, 2 equations in 2 unknowns) — solve it the same way.
3. **Back-substitute** upward to recover every variable.

We work the lecture's example:

$$ \begin{cases} a + b + 2c = 12 \\ 3a - 3b - c = 3 \\ 2a - b + 6c = 24 \end{cases} $$

In [1]:
import numpy as np
from fractions import Fraction as F   # exact arithmetic, to match the slides' fractions

def show(rows, names="abc"):
    for r in rows:
        terms = " + ".join(f"{r[i]}{names[i]}" for i in range(len(r) - 1))
        print(f"  {terms} = {r[-1]}")

## 1. Eliminate $a$

**Divide each row by its coefficient of $a$** so every equation starts with $a$:

$$ a + b + 2c = 12 \qquad a - b - \tfrac{1}{3}c = 1 \qquad a - \tfrac{1}{2}b + 3c = 12 $$

**Subtract the first equation** from the other two to remove $a$:

$$ a + b + 2c = 12 \qquad -2b - \tfrac{7}{3}c = -11 \qquad -\tfrac{3}{2}b + c = 0 $$

In [2]:
# rows as [a, b, c, rhs] with exact fractions
rows = [[F(1), F(1), F(2), F(12)],
        [F(3), F(-3), F(-1), F(3)],
        [F(2), F(-1), F(6), F(24)]]

# divide each row by its coefficient of a
rows = [[x / r[0] for x in r] for r in rows]
print("After dividing by coefficient of a:"); show(rows)

# subtract row 0 from rows 1 and 2 -> a eliminated there
for i in (1, 2):
    rows[i] = [rows[i][k] - rows[0][k] for k in range(4)]
print("\nAfter eliminating a:"); show(rows)

After dividing by coefficient of a:
  1a + 1b + 2c = 12
  1a + -1b + -1/3c = 1
  1a + -1/2b + 3c = 12

After eliminating a:
  1a + 1b + 2c = 12
  0a + -2b + -7/3c = -11
  0a + -3/2b + 1c = 0


## 2. A smaller 2×2 system in $b, c$ — eliminate $b$

Ignoring the (now solved-for-later) first row, rows 2 and 3 form a system in $b$ and $c$. Divide each by its coefficient of $b$:

$$ b + \tfrac{7}{6}c = \tfrac{11}{2} \qquad b - \tfrac{2}{3}c = 0 $$

Subtract to remove $b$, which isolates $c$:

$$ -\tfrac{11}{6}c = -\tfrac{11}{2} \;\Rightarrow\; c = 3. $$

In [3]:
# work on the lower 2x2 block (variables b, c, rhs) from rows[1], rows[2]
sub = [rows[1][1:], rows[2][1:]]          # drop the a-column (all zeros)
sub = [[x / r[0] for x in r] for r in sub]  # divide by coefficient of b
print("After dividing by coefficient of b:"); show(sub, names="bc")

sub[1] = [sub[1][k] - sub[0][k] for k in range(3)]  # eliminate b
c = sub[1][2] / sub[1][1]
print("\nIsolated row:", sub[1], " ->  c =", c)

After dividing by coefficient of b:
  1b + 7/6c = 11/2
  1b + -2/3c = 0

Isolated row: [Fraction(0, 1), Fraction(-11, 6), Fraction(-11, 2)]  ->  c = 3


## 3. Back-substitute

- Into $b + \tfrac{7}{6}c = \tfrac{11}{2}$ with $c=3$: $\;b = \tfrac{11}{2} - \tfrac{7}{2} = 2.$
- Into $a + b + 2c = 12$ with $b=2, c=3$: $\;a = 12 - 2 - 6 = 4.$

$$ \boxed{a = 4, \quad b = 2, \quad c = 3} $$

In [4]:
b = sub[0][2] - sub[0][1] * c            # from b + 7/6 c = 11/2
a = rows[0][3] - rows[0][1] * b - rows[0][2] * c   # from a + b + 2c = 12
print(f"a = {a}, b = {b}, c = {c}")

a = 4, b = 2, c = 3


In [5]:
# Cross-check with numpy.
A = np.array([[1.0, 1.0, 2.0], [3.0, -3.0, -1.0], [2.0, -1.0, 6.0]])
d = np.array([12.0, 3.0, 24.0])
print("numpy solution [a, b, c] =", np.linalg.solve(A, d))
print("determinant =", round(np.linalg.det(A)), "(non-zero -> non-singular -> unique solution)")

numpy solution [a, b, c] = [4. 2. 3.]
determinant = -33 (non-zero -> non-singular -> unique solution)


## 4. The pattern (row echelon form)

Elimination drives the system to a **triangular** (row-echelon) shape, where each equation has one fewer variable than the one above:

$$
\begin{aligned}
a + b + 2c &= 12 \\
b + \tfrac{7}{6}c &= \tfrac{11}{2} \\
c &= 3
\end{aligned}
$$

From the bottom row you read off $c$, then back-substitute up. This generalises to any number of variables: **eliminate down to a triangle, then back-substitute up** — exactly what `np.linalg.solve` does internally (LU decomposition).

## 5. Takeaways

- More variables ⇒ same method, applied **recursively**: remove the first variable, then solve the smaller system that remains.
- Each step normalises a **pivot** to 1 and subtracts to clear that variable from the other rows.
- The system reaches a **triangular (row-echelon) form**; **back-substitution** then recovers every unknown.
- Geometrically, three equations in three unknowns are three **planes**; a non-singular system meets at a single point ($a=4, b=2, c=3$).

*Companion:* [solving non-singular systems](./C1_W2_L1_solving_non_singular_systems.ipynb) (the 2-variable version) · [solving singular systems](./C1_W2_L1_solving_singular_systems.ipynb).